## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [ ]:
propuski = titanic_data.isnull().sum()
propuski

### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [ ]:
n_strok = titanic_data.shape[0]
n_stolbtsov = titanic_data.shape[1]

porog_stolbtsov = n_strok / 2
titanic_chistiy = titanic_data.dropna(axis="columns", thresh=porog_stolbtsov)

porog_strok = titanic_chistiy.shape[1] / 2
titanic_chistiy = titanic_chistiy.dropna(thresh=porog_strok).copy()

print(f"размер таблицы(я кстати даже тут зафейлил): {titanic_chistiy.shape}")
titanic_chistiy.isnull().sum()

### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [ ]:
vozrast_est = titanic_chistiy["age"].notnull()

muzhchiny = titanic_chistiy["who"] == "man"
zhenschiny = titanic_chistiy["who"] == "woman"
deti = titanic_chistiy["who"] == "child"

vozrast_muzhchin = titanic_chistiy.loc[vozrast_est & muzhchiny, "age"]
vozrast_zhenschin = titanic_chistiy.loc[vozrast_est & zhenschiny, "age"]
vozrast_detey = titanic_chistiy.loc[vozrast_est & deti, "age"]

mediana_muzhchin = round(vozrast_muzhchin.median())
mediana_zhenschin = round(vozrast_zhenschin.median())
mediana_detey = round(vozrast_detey.median())

print(f"срзнач мужчин: {mediana_muzhchin}")
print(f"женщин: {mediana_zhenschin}")
print(f"детей: {mediana_detey}")

vozrast_pusto = titanic_chistiy["age"].isnull()

titanic_chistiy.loc[vozrast_pusto & muzhchiny, "age"] = mediana_muzhchin
titanic_chistiy.loc[vozrast_pusto & zhenschiny, "age"] = mediana_zhenschin
titanic_chistiy.loc[vozrast_pusto & deti, "age"] = mediana_detey

print(f"пропуски в столбце age: {titanic_chistiy['age'].isnull().sum()}")
titanic_chistiy.isnull().sum()

### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [ ]:
n_stolbtsov_teper = titanic_chistiy.shape[1]
min_znacheniy = n_stolbtsov_teper - 1
titanic_chistiy = titanic_chistiy.dropna(thresh=min_znacheniy)

vse_propuski = titanic_chistiy.isnull().sum().sum()
print(f"размер: {titanic_chistiy.shape}")
print(f"пропуски: {vse_propuski}")

### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [ ]:
goroda = titanic_chistiy["embark_town"].value_counts()

glavniy_gorod = goroda.index[0]

print(f"самый популярный город: {glavniy_gorod}")
goroda

### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [ ]:
vyzhilo = titanic_chistiy["survived"].sum()
vsego = titanic_chistiy.shape[0]

dolya = vyzhilo / vsego
protcent = round(dolya * 100, 2)

print(f"сыжившие: {vyzhilo} из {vsego}")
print(f"процент: {protcent}%")

### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [ ]:
vyzhivshiye = titanic_chistiy[titanic_chistiy["survived"] == 1]

vyzhivshiye_po_gorodam = vyzhivshiye["embark_town"].value_counts()
vyzhivshiye_po_gorodam

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [ ]:
klassy = titanic_chistiy["class"].unique()
slovar_protsentov = {}

for klass in klassy:
    gruppa = titanic_chistiy[titanic_chistiy["class"] == klass]

    vsego_v_klasse = gruppa.shape[0]
    vyzhilo_v_klasse = gruppa["survived"].sum()

    protcent = round(vyzhilo_v_klasse / vsego_v_klasse * 100, 2)
    slovar_protsentov[klass] = protcent

protcenty_po_klassam = pd.Series(slovar_protsentov)
protcenty_po_klassam

### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [ ]:
bogatiye = titanic_chistiy[titanic_chistiy["fare"] >= 100]

vsego_bogatikh = bogatiye.shape[0]
vyzhilo_bogatikh = bogatiye["survived"].sum()

protcent_bogatikh = round(vyzhilo_bogatikh / vsego_bogatikh * 100, 2)
print(f"Богатые: {vsego_bogatikh}")
print(f"выжило: {vyzhilo_bogatikh}")
print(f"Процент: {protcent_bogatikh}%")

### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [ ]:
rebyata = titanic_chistiy["who"] == "child"
odni = titanic_chistiy["alone"]

deti_odni = titanic_chistiy[rebyata & odni]

print(f"дети в одиночку: {deti_odni.shape[0]}")

почему я не был на титанике :((